In [0]:
from pyspark.sql.functions import *
import requests
import base64
import pandas as pd

In [0]:

TEMPLATE_FOLDER = "/Volumes/xliidw_dev_lpl/application/adhoc_files/notebook_automation"

CATALOG_NAME = "xliidw_dev_lpl"

TKN = ""
API_URL = "https://adb-6031231472145779.19.azuredatabricks.net/api/2.0/workspace/import"
HEADERS = {
    "Authorization": f"Bearer {TKN}",
    "Content-Type": "application/json"
}

ONE_TO_ONE_TEMPLATE = f"{TEMPLATE_FOLDER}/Populate_xliidw_one_to_one_template.py"
COMPLEX_TEMPLATE = f"{TEMPLATE_FOLDER}/Populate_xliidw_complex_template.py"
CORE_LOGIC_TEMPLATE = f"{TEMPLATE_FOLDER}/core_logic_template.py"

INFORMATICA_COMPONENT_EXCEL_PATH = f"{TEMPLATE_FOLDER}/IDW_All_Component_V3.xlsx"

#FAME_ODS_SCHEMA_NAME = 'frameods'

WORKSPACE_BASE_FOLDER = '/Workspace/XLIIDW/XLIIDW_LOAD'

ENV_CONFIG_DB_NAME = "application"

In [0]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType

pandas_df = pd.read_excel(INFORMATICA_COMPONENT_EXCEL_PATH, sheet_name='Active_Infa_components', dtype=str)

schema = StructType([
    StructField("Folder_Name", StringType(), True),
    StructField("Workflow_Name", StringType(), True),
    StructField("Session_Name", StringType(), True),
    StructField("Mapping_Name", StringType(), True),
    StructField("is_one_one_mapping", StringType(), True),
    StructField("Source_connection", StringType(), True),
    StructField("Source_Database_Name", StringType(), True),
    StructField("Source_Type", StringType(), True),
    StructField("Source_object_name", StringType(), True),
    StructField("Source_object_synonym", StringType(), True),
    StructField("Target_connection", StringType(), True),
    StructField("Target_Database_Name", StringType(), True),
    StructField("Target_Type", StringType(), True),
    StructField("Target_object_name", StringType(), True)
])

df_frameods_all_active_infa_components = spark.createDataFrame(pandas_df, schema)

for column in df_frameods_all_active_infa_components.columns:
    df_frameods_all_active_infa_components = df_frameods_all_active_infa_components.withColumn(column, trim(df_frameods_all_active_infa_components[column]))

df_frameods_all_active_infa_components.createOrReplaceTempView("active_infa_components")

spark.sql("select * from active_infa_components").display()


In [0]:
query = f"""SELECT Folder_Name,
Workflow_Name,
Session_Name,
Mapping_Name,
'{CATALOG_NAME}' as Source_catalog,
Source_Database_Name as Source_db,
Source_object_name as Source,
'{CATALOG_NAME}' as Target_catalog,
Target_Database_Name as Target_db,
Target_object_name as Target,
CASE
  WHEN is_one_one_mapping = 'Y'
  THEN 'Y'
  ELSE 'N'
END AS is_one_one_mapping
FROM active_Infa_components where Mapping_Name <> 'Disabled' """

fullMappingsDetails = spark.sql(query)
fullMappingsDetails.display()



In [0]:
# **** TEST CASES ****
#PROD_FRAME one to one - m_Populate_Unallocated_Cash
#PROD_FRAME complex - m_SD_dtpDocType_to_Document_Type
#PROD_ODS one to one synonym - m_SD_inmInsMkt_to_Insurance_Market source swanreprestore
#PROD_ODS one to one NON synonym - m_Populate_Object_Note_Content_PDO source is frameods
#PROD_ODS complex synonym - m_Populate_Base_Financial_Codes
#PROD_ODS complex NON synonym - m_Populate_Object_Note_PDO


mappingsDetails = fullMappingsDetails#.filter(" mapping_name in ( 'm_Populate_Unallocated_Cash', 'm_SD_dtpDocType_to_Document_Type', 'm_SD_inmInsMkt_to_Insurance_Market', 'm_Populate_Object_Note_Content_PDO', 'm_Populate_Base_Financial_Codes', 'm_Populate_Object_Note_PDO')")

mappingsDetails.display()

In [0]:
# Master mapping configuration
MASTER_MAPPING = {'xli_staging_procede_102': {'mapping_folder':'XLI_PROCEDE', 'category':'XLI_PROCEDE_Population'}}

In [0]:
def write_script_to_file(script, file_name, mode='a+'):    
    try:
        print(script)
        file = open(f"GENERATED_SCRIPTS/{file_name}", mode)
        file.write(script)
        file.write("\n")
        file.close

    except Exception as e:
        print(f"An error occurred: {e}")

def create_folder_if_not_exists(folder_path):
    data = {"path": folder_path}
    folder_status = requests.get("https://adb-6031231472145779.19.azuredatabricks.net/api/2.0/workspace/get-status", headers=HEADERS, json=data)
    if folder_status.status_code==404:
        response=requests.post("https://adb-6031231472145779.19.azuredatabricks.net/api/2.0/workspace/mkdirs", headers=HEADERS, json=data)
        if response.status_code==200:
            print(f"{folder_path} folder created successfully")
            return True
    elif folder_status.status_code==200:
        return True

def load_notebook_template(template_path):
    """Load notebook template from the given file path."""
    with open(template_path, 'r') as f:
        return f.read()

def create_notebook_content(template, replacements):
    """Create the notebook content by replacing placeholders in the template."""
    for placeholder, value in replacements.items():
        template = template.replace(placeholder, value)
    return base64.standard_b64encode(template.encode('utf-8')).decode('utf-8')

def check_table_existence(catalog, db, table):
    databases = [row.databaseName.upper() for row in spark.sql(f"SHOW DATABASES IN {catalog}").collect()]
    
    if db.upper() not in databases:
        return False

    tables = [row.tableName.upper() for row in spark.sql(f"SHOW TABLES IN {catalog}.{db}").collect()]
    
    return table.upper() in tables


def insert_job_config(job_identifier, job_group_identifier, notebook_path):
    """Prepare the SQL insert statement for the job configuration."""
    insert_statement = f"""INSERT INTO [config].[t_batch_load_job_configs]([job_identifier],[job_group_identifier],
                [job_seq_no],[job_notebook_path],[job_notebook_param],[logging_mode],[is_active])
                VALUES ('{job_identifier}', '{job_group_identifier}', 1, '{notebook_path}', 
                N'{{"job_config_identifier":"{job_identifier}"}}', 'INFO', 'Y');"""
    write_script_to_file(insert_statement, "7A_insert_t_batch_load_job_configs.sql")
    return insert_statement

def create_notebook(notebook_folder, trimmed_mapping_name, template_path, target_notebook_path, replacements):
    """Create a notebook in Databricks using the specified template and replacements."""
    try:
        # Load the notebook template
        notebook_template = load_notebook_template(template_path)

        # Create the notebook content
        notebook_content = create_notebook_content(notebook_template, replacements)

        # Create the notebook in Databricks
        data = {
            "format": "SOURCE",
            "path": target_notebook_path,
            "language": "PYTHON",
            "content": notebook_content,
            "overwrite": True
        }

        response = requests.post(API_URL, headers=HEADERS, json=data)
        print(f"response - {response}")
        if response.status_code == 200:
            print(f"Notebook '{target_notebook_path}' created successfully.")
            return "CREATED", "created successfully"
        else:
            raise Exception(f"Failed to create notebook. Status code: {response.status_code}. Message: {response.text}")

    except Exception as e:
        print(f"Exception occurred while generating notebook {target_notebook_path}:\n EXCEPTION: {e}")
        return "FAILED", str(e)
    
def generate_databricks_load_config_insert_statement(config_identifier_text, config_name_value_mappings):
    insert_statements = []

    for name, value in config_name_value_mappings.items():
        if name == "param_tbl_col_mapping" or name == "param_tgt_write_options":
            is_expression_ind = 'N'
            value = value.replace("'", '"')
        else:
            is_expression_ind = 'N'
        
        sql_statement = f"""INSERT INTO [config].[t_databricks_load_config] ([config_identifier_text], [config_name], [config_value], [is_expression_ind]) VALUES ('{config_identifier_text}','{name}','{value}','{is_expression_ind}');"""

        insert_statements.append(sql_statement.strip())
    
    return insert_statements

def create_parameter_entries_of_the_job(job_identifier, is_one_one_mapping, replacements):
    """Create the parameter entries for the job."""

    write_script_to_file(f"\n--Creating parameter entries for the Job : {job_identifier}\n", "7A_insert_t_databricks_load_configs.sql")
    delete_script = f"DELETE FROM [config].[t_databricks_load_config] WHERE [config_identifier_text] = '{job_identifier}';"
    write_script_to_file(f"--{delete_script}\n", "7A_insert_t_databricks_load_configs.sql")


    parameter_insert_scripts = []
    parameter_insert_script = ""
    mapping_default_parameter_mapping = {}
    one_to_one_mapping_default_parameter_text = """{
    "param_src_db_name":"<<source_catalog>>.<<source_db>>",
    "param_tgt_db_name":"<<target_catalog>>.<<target_db>>",
    "param_tgt_tbl_name":"<<target_catalog>>.<<target_db>>.<<target_table_name>>",
    "param_tbl_nme_<<source_db>>_<<source_table_name>>":"<<source_catalog>>.<<source_db>>.<<source_table_name>>",
    "param_tbl_nme_<<target_db>>_<<target_table_name>>":"<<target_catalog>>.<<target_db>>.<<target_table_name>>",
    "param_tbl_col_mapping":"<<column_mapping>>",
    "one_line_spark_code_for_transformation":"",
    "param_src_balance_sql":"select count(1) as count from <<source_catalog>>.<<source_db>>.<<source_table_name>>",
    "param_tgt_balance_sql":"select count(1) as count from <<target_catalog>>.<<target_db>>.<<target_table_name>>",
    "param_tgt_write_mode":"overwrite",
    "param_tgt_partition_columns":"[]",
    "param_tgt_write_options":"{'mergeSchema': 'true'}",
    "param_tgt_vaccum_retention_period_hours":'168',
    "param_tgt_schema":"None"}"""

    complex_mapping_default_parameter_text = """{
                            "param_src_db_name":"<<source_catalog>>.TODO_source_db_name",
                            "param_tgt_db_name":"<<target_catalog>>.TODO_target_db_name",
                            "param_tgt_tbl_name":"TODO_tgt_tbl_name",
                            "param_src_balance_sql":"select 0 as count",
                            "param_tgt_balance_sql":"select 0 as count",
                            "param_tgt_write_mode":"overwrite",
                            "param_tgt_partition_columns":"[]",
                            "param_tgt_write_options":"{'mergeSchema': 'true'}",
                            "param_tgt_vaccum_retention_period_hours":'168',
                            "param_tgt_schema":"None"
    }"""

    if is_one_one_mapping == "Y":
        mapping_default_parameter_text = one_to_one_mapping_default_parameter_text
    else:
        mapping_default_parameter_text = complex_mapping_default_parameter_text

    for replacement_key, replacement_value in replacements.items():
        #print(f" replacement_key - {replacement_key}; replacement_value - {replacement_value}")
        mapping_default_parameter_text = mapping_default_parameter_text.replace(replacement_key, replacement_value)
    
    print(mapping_default_parameter_text)
    mapping_default_parameter_mapping = eval(mapping_default_parameter_text)

    parameter_insert_scripts = generate_databricks_load_config_insert_statement(job_identifier, mapping_default_parameter_mapping)

    for script in parameter_insert_scripts:
        write_script_to_file(script, "7A_insert_t_databricks_load_configs.sql")
        print(script)


In [0]:
#project_base_folder = '/Workspace/FRAME_ODS/DATA_LOAD'

# if '/AUTOMATION/GENERATED_NOTEBOOKS' not in WORKSPACE_BASE_FOLDER:
#     raise Exception("The base folder must be a user folder !!! .")

# DataFrame to store job config inserts and notebook creation status
job_config_table_inserts = [["", ""]]
notebook_creation_status = [["folder_name", 'mapping_name', 'status', 'message']]

write_script_to_file("\n--***************CONFIG ENTRIES FOR mappings***************\n\n", "7A_insert_t_batch_load_job_configs.sql", mode='w')
write_script_to_file("\n--***************PARMETER ENTRIES FOR mappings***************\n\n", "7A_insert_t_databricks_load_configs.sql", mode='w')

# Main execution block
for mapping_details in mappingsDetails.collect():
    folder_name = "DUMMY" if mapping_details[0] is None else mapping_details[0].strip()
    workflow_name = "" if mapping_details[1] is None else mapping_details[1].strip()
    mapping_name = "" if mapping_details[3] is None else mapping_details[3].strip()
    source_catalog = "" if mapping_details[4] is None else mapping_details[4].strip()
    source_db = "" if mapping_details[5] is None else mapping_details[5].strip()
    source_table_name = "" if mapping_details[6] is None else mapping_details[6].strip()
    target_catalog = "" if mapping_details[7] is None else mapping_details[7].strip()
    target_db = "" if mapping_details[8] is None else mapping_details[8].strip()
    target_table_name = "" if mapping_details[9] is None else mapping_details[9].strip()
    is_one_to_one_mapping = "N" if mapping_details[10] is None else mapping_details[10].strip()

    # Prepare notebook properties
    trimmed_mapping_name = mapping_name[2:] if mapping_name.startswith("m_") else mapping_name
    trimmed_workflow_name = workflow_name[3:] if workflow_name.startswith("wf_") else workflow_name
    
    notebook_base_folder = MASTER_MAPPING[folder_name]['mapping_folder']
    
    full_qualified_notebook_folder = f"{WORKSPACE_BASE_FOLDER}/{notebook_base_folder}"
    full_qualified_notebook_folder_core_logic = f"{full_qualified_notebook_folder}/core_logic"
    full_qualified_notebook_folder_template = f"{full_qualified_notebook_folder}/template"
    
    job_identifier = f"{notebook_base_folder}-{trimmed_mapping_name}"
    #job_identifier = f"{notebook_base_folder}-{trimmed_workflow_name}-{trimmed_mapping_name}"
    #job_group_identifier = f"{notebook_base_folder}-{trimmed_workflow_name}-group"
    job_group_identifier = f"{notebook_base_folder}-group"

    # Prepare target notebook paths
    main_notebook_path = f"{full_qualified_notebook_folder}/{trimmed_mapping_name}"
    core_logic_notebook_path = f"{full_qualified_notebook_folder_core_logic}/{trimmed_mapping_name}"
    template_notebook_path = f"{full_qualified_notebook_folder_template}/{trimmed_mapping_name}"

    create_folder_if_not_exists(full_qualified_notebook_folder)
    create_folder_if_not_exists(full_qualified_notebook_folder_core_logic)
    create_folder_if_not_exists(full_qualified_notebook_folder_template)

    # Initialize column mapping
    column_mapping = "{}"

    # Check for one-to-one mapping and load column mapping
    if is_one_to_one_mapping == 'Y':
        if (check_table_existence(source_catalog, source_db, source_table_name) and 
            check_table_existence(target_catalog, target_db, target_table_name)):
            
            # Load source and target table schemas
            source_schema = spark.table(f"{source_catalog}.{source_db}.{source_table_name}").schema.fieldNames()
            target_schema = spark.table(f"{target_catalog}.{target_db}.{target_table_name}").schema.fieldNames()

            # Create a dictionary mapping source columns to target columns
            column_mapping = str(dict(zip(source_schema, target_schema)))

    # Prepare replacements for the notebook
    replacements = {
        '<<job_config_identifier>>': job_identifier,
        '<<batch_set_category>>': MASTER_MAPPING[folder_name]['category'],
        '<<batch_set_src_sys_name>>': f"SSNM-{job_group_identifier}",
        '<<batch_set_src_sys_cd>>': f"SSCD-{job_group_identifier}",
        '<<job_identifier>>': job_identifier,
        '<<env_config_table>>': f"{CATALOG_NAME}.{ENV_CONFIG_DB_NAME}.t_environment_config",
        '<<logging_mode>>': "DEBUG",
        '<<source_catalog>>': source_catalog,
        '<<target_catalog>>': target_catalog,
        '<<source_db>>': source_db.strip(),
        '<<target_db>>': target_db.strip(),
        '<<source_table_name>>': source_table_name.strip(),
        '<<target_table_name>>': target_table_name.strip(),
        '<<column_mapping>>': column_mapping,
        '<<core_logic_notebook_path>>': core_logic_notebook_path
    }

    # Create the main notebook based on mapping type
    if is_one_to_one_mapping == 'Y':
        print(f"Creating one-to-one notebook for {mapping_name}")
        #status, message = ('success', 'created')
        status, message = create_notebook(full_qualified_notebook_folder, trimmed_mapping_name, ONE_TO_ONE_TEMPLATE, main_notebook_path, replacements)
        # Insert job config for the main notebook
        job_config_table_insert = insert_job_config(job_identifier, job_group_identifier, main_notebook_path)
        job_config_table_inserts.append([job_identifier, job_config_table_insert])
    else:
        # Create the complex notebook - in MAIN folder itself
        #status, message = create_notebook(full_qualified_notebook_folder, trimmed_mapping_name, COMPLEX_TEMPLATE, main_notebook_path, replacements)

        # Create the complex notebook in tempate folder for future merge with core logic
        #status, message = ('success', 'created')
        status, message = create_notebook(full_qualified_notebook_folder_template, trimmed_mapping_name, COMPLEX_TEMPLATE, template_notebook_path, replacements)

        # Insert job config for the main notebook
        job_config_table_insert = insert_job_config(job_identifier, job_group_identifier, main_notebook_path)
        job_config_table_inserts.append([job_identifier, job_config_table_insert])
        
        # Create the core logic child notebook using the same function, no job config entry needed
        #child_status, child_message = ('success', 'created')
        child_status, child_message = create_notebook(full_qualified_notebook_folder_core_logic, trimmed_mapping_name, CORE_LOGIC_TEMPLATE, core_logic_notebook_path, {})
        notebook_creation_status.append([folder_name, mapping_name, child_status, child_message])

    create_parameter_entries_of_the_job(job_identifier, is_one_to_one_mapping, replacements)

    notebook_creation_status.append([folder_name, mapping_name, status, message])

# Create DataFrames for Spark

config_insert_df = spark.createDataFrame(job_config_table_inserts, ['job_config_identifier', 'job_config_table_insert'])
config_insert_df.display()

notebook_creation_status_df = spark.createDataFrame(notebook_creation_status, ['folder_name', 'mapping_name', 'notebook_creation_status', 'notebook_creation_message'])
notebook_creation_status_df.display()


In [0]:
%sh ls /Volumes/xliidw_dev_lpl/application/adhoc_files/notebook_automation/

In [0]:

write_script_to_file("\n--***************insert script for progressing bacth set for all workflows*************** \n\n", "7A_insert_t_batch_set_entry.sql", mode='w')
write_script_to_file("\n--***************insert script of t_batch_load_job_group_configs table for all workflows*************** \n\n", "7A_insert_t_batch_load_job_group_configs_entry.sql", mode='w')

for mapping_details in mappingsDetails.select('Folder_Name', 'Workflow_Name').distinct().collect():
    folder_name = mapping_details[0].strip()
    workflow_name = mapping_details[1]
    trimmed_workflow_name = workflow_name[3:] if workflow_name.startswith("wf_") else workflow_name
    notebook_folder = MASTER_MAPPING[folder_name]['mapping_folder']
    job_group_identifier = f"{notebook_folder}-group"

    batch_set_insert_statement = f"""INSERT INTO [batch].[t_batch_set]
           ([batch_set_category],[batch_set_src_sys_nme],[batch_set_src_sys_cd],[batch_set_cfg_value],[batch_set_strt_tm],[batch_set_end_tm],[batch_set_sts_ind],[batch_set_sts_descr],[batch_force_run_ind])
     VALUES
           ('{MASTER_MAPPING[folder_name]['category']}','{f"SSNM-{job_group_identifier}"}','{f"SSCD-{job_group_identifier}"}','F',SYSDATETIME(),NULL,'P','Progress','N');"""

    batch_load_job_group_configs_insert_statement = f"""INSERT INTO [config].[t_batch_load_job_group_configs] ([job_group_identifier],[trigger_identifier],[job_group_details],[parallel_job_no],[cluster_node_type],[num_of_worker_nodes],[restart_failed_jobs_flag],[is_active],[cluster_run_time],[cluster_configs]) VALUES ('{job_group_identifier}','{notebook_folder}','{{"mail_enabled_flag":"N"}}',10,'Standard_DS13_v2','2:50','N','Y','15.4.x-scala2.12',NULL);"""

    
    write_script_to_file(batch_set_insert_statement, "7A_insert_t_batch_set_entry.sql")
    write_script_to_file(batch_load_job_group_configs_insert_statement, "7A_insert_t_batch_load_job_group_configs_entry.sql")
    print(batch_set_insert_statement)